# Train DeepLabV3 on the CMP Facade Database

Run this notebook on Google Colab with a GPU runtime (Runtime -> Change runtime type -> GPU).

It fine-tunes a pretrained DeepLabV3 (ResNet-50 backbone) on the CMP Facade Database (606 rectified facade photos, 12-class pixel labels) to produce a `best_model.pt` checkpoint usable by `src/evaluate.py` and `app/streamlit_app.py`.

## 1. Setup

Clone the repo (or upload `src/` if working from a zip) so the training script can `import` the shared `src/` modules instead of duplicating logic.

In [ ]:
!pip install -q datasets huggingface_hub

!git clone https://github.com/VickeyAryan/building-envelope-vision.git
%cd building-envelope-vision

# If you'd rather upload the project as a zip instead of cloning, comment out
# the two lines above and use this instead:
# !unzip -q building-envelope-vision.zip
# %cd building-envelope-vision

import sys
sys.path.insert(0, 'src')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Load the dataset

`Xpitfire/cmp_facade` ships `train` (378), `test` (114), and `eval` (114) splits, RGB photo + palette-indexed class mask per example.

In [ ]:
from data_loader import get_datasets, CLASS_NAMES, NUM_CLASSES

IMAGE_SIZE = 384
train_ds, val_ds, test_ds = get_datasets(image_size=IMAGE_SIZE)
print('train/val/test sizes:', len(train_ds), len(val_ds), len(test_ds))
print('classes:', CLASS_NAMES)

### Sanity-check a sample

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from data_loader import denormalize, PALETTE

img_t, mask_t = train_ds[0]
img_disp = denormalize(img_t).permute(1, 2, 0).numpy()
mask_np = mask_t.numpy()

color_mask = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
for idx, rgb in enumerate(PALETTE):
    color_mask[mask_np == idx] = rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_disp)
axes[0].set_title('Image')
axes[0].axis('off')
axes[1].imshow(color_mask)
axes[1].set_title('Ground-truth mask')
axes[1].axis('off')
plt.show()

## 3. Train

Uses `src/train.py`'s `train_one_epoch` + `src/evaluate.py`'s `evaluate` directly, so the notebook and the CLI (`python src/train.py`) share identical training/eval logic.

In [ ]:
import time
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader

from model import build_model
from train import train_one_epoch
from evaluate import evaluate

EPOCHS = 60
BATCH_SIZE = 8
LR = 1e-4
OUTPUT_PATH = Path('models/best_model.pt')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model = build_model(num_classes=NUM_CLASSES, pretrained=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

history = []
best_miou = -1.0
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_results = evaluate(model, val_loader, device)
    elapsed = time.time() - start

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_mean_iou': val_results['mean_iou'],
        'val_pixel_acc': val_results['pixel_accuracy'],
    })
    print(f"Epoch {epoch}/{EPOCHS} | train_loss={train_loss:.4f} "
          f"| val_mIoU={val_results['mean_iou']:.4f} "
          f"| val_pixel_acc={val_results['pixel_accuracy']:.4f} | {elapsed:.1f}s")

    if val_results['mean_iou'] > best_miou:
        best_miou = val_results['mean_iou']
        torch.save(model.state_dict(), OUTPUT_PATH)
        print(f"  -> new best (mIoU={best_miou:.4f}), saved to {OUTPUT_PATH}")

print(f"Training complete. Best val mIoU: {best_miou:.4f}")

### Training curves

In [ ]:
epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [h['train_loss'] for h in history])
axes[0].set_title('Train loss')
axes[0].set_xlabel('epoch')
axes[1].plot(epochs, [h['val_mean_iou'] for h in history], label='mean IoU')
axes[1].plot(epochs, [h['val_pixel_acc'] for h in history], label='pixel acc')
axes[1].set_title('Validation metrics')
axes[1].set_xlabel('epoch')
axes[1].legend()
plt.show()

## 4. Evaluate on the held-out test split

In [ ]:
from model import load_checkpoint

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
best_model = load_checkpoint(str(OUTPUT_PATH), num_classes=NUM_CLASSES, device=device)
test_results = evaluate(best_model, test_loader, device)

print(f"Test pixel accuracy: {test_results['pixel_accuracy']:.4f}")
print(f"Test mean IoU: {test_results['mean_iou']:.4f}")
print('Per-class IoU:')
for name, val in test_results['per_class_iou'].items():
    print(f"  {name:12s}: {'n/a' if val is None else f'{val:.4f}'}")

## 5. Qualitative check: predicted mask + facade metrics on a test image

In [ ]:
from metrics import compute_all_metrics

img_t, mask_t = test_ds[0]
with torch.no_grad():
    logits = best_model(img_t.unsqueeze(0).to(device))['out']
pred_mask = logits.argmax(dim=1).squeeze(0).cpu().numpy()

img_disp = (denormalize(img_t).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
pred_color = np.zeros((*pred_mask.shape, 3), dtype=np.uint8)
for idx, rgb in enumerate(PALETTE):
    pred_color[pred_mask == idx] = rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img_disp)
axes[0].set_title('Image')
axes[0].axis('off')
axes[1].imshow(pred_color)
axes[1].set_title('Predicted mask')
axes[1].axis('off')
plt.show()

print(compute_all_metrics(img_disp, pred_mask))

## 6. Download the checkpoint

Copy `models/best_model.pt` back to your local `models/` directory so `app/streamlit_app.py` can use it.

In [ ]:
try:
    from google.colab import files
    files.download(str(OUTPUT_PATH))
except ImportError:
    print(f"Not running in Colab -- checkpoint already saved locally at {OUTPUT_PATH}")